# RaiFlow — EU AI Act Compliance Walkthrough

This notebook walks through every feature of the RaiFlow compliance gate.

**What you'll learn:**
1. Scaffold a compliance manifest with `raiflow init`
2. Run static compliance checks across all pipeline stages
3. Use `--target` for Article 14 human oversight endpoint scanning
4. Use `--threshold` to override per-rule pass thresholds
5. Run LLM-based semantic checks with `--enable-llm-checks` (mocked — no Ollama needed)
6. Understand the `evaluation_mode` field in the compliance report
7. Generate pytest compliance tests with `raiflow generate-tests`
8. Integrate with GitHub Actions

**Articles covered:** EU AI Act Articles 9–14

---
**Prerequisites:** `pip install raiflow`

In [ ]:
%pip install raiflow --quiet

## 1. `raiflow init` — scaffold your project

`raiflow init` scans your project for AI framework imports (LangChain, OpenAI, Ollama, LlamaIndex, Transformers, FastAPI, etc.), infers a risk level, and writes:
- `raiflow.yaml` — your compliance manifest
- `.github/workflows/rai-compliance.yml` — a ready-to-use GitHub Actions workflow

It also runs an interactive LLM setup wizard to configure your backend for `--enable-llm-checks`.

```bash
raiflow init
raiflow init --directory ./my-ai-project
raiflow init --force   # overwrite existing files
```

For this notebook we create the manifest and supporting files inline:

In [ ]:
from pathlib import Path

# Risk assessment document (Article 9)
Path('demo_risk_assessment.md').write_text("""
# Risk Assessment — my-rag-app
## Identified Risks
- Hallucination in retrieval-augmented responses
- Bias in training corpus affecting underrepresented groups
- Misuse for generating misleading content
## Mitigation Measures
- Retrieval grounding with source citations
- Regular bias audits on dataset and outputs
- Content moderation layer on all outputs
## Lifecycle Coverage
Risks assessed at design, development, deployment, and decommissioning stages.
## Vulnerable Groups
Special consideration for users under 18 and non-native language speakers.
""")

# Technical documentation (Article 11 — Annex IV)
Path('demo_tech_doc.md').write_text("""
# Technical Documentation — my-rag-app (Annex IV)
## (a) General System Description
A retrieval-augmented generation system for enterprise knowledge management.
Intended purpose: answer employee queries using internal documentation.
## (b) Training Data and Data Governance
Base model: Llama 3.1. Fine-tuned on 50k internal Q&A pairs.
Data governance: annotation guidelines v2.1, bias review completed 2025-01.
## (c) Performance Metrics
- Accuracy on held-out eval set: 87.3%
- Hallucination rate: <3% on factual queries
- Latency p95: 1.2s
## (d) Known Limitations
- Does not handle queries outside the indexed document corpus
- Not suitable for medical or legal advice
## (e) Human Oversight Measures
- All responses include source citations for human verification
- Override endpoint /api/override allows operators to halt the system
- Weekly human review of flagged low-confidence responses
""")

# Source file with override endpoints (Article 14)
Path('demo_app.py').write_text("""
from fastapi import FastAPI
app = FastAPI()

@app.post('/api/override')
def override_model():
    return {'status': 'model overridden'}

@app.post('/api/halt')
def halt_model():
    return {'status': 'model halted'}
""")

# The compliance manifest
Path('demo_raiflow.yaml').write_text("""
system_name: \"my-rag-app\"
risk_level: \"high\"
eu_ai_act_articles:
  - \"Article 9\"
  - \"Article 10\"
  - \"Article 11\"
  - \"Article 12\"
  - \"Article 13\"
  - \"Article 14\"

model_metadata:
  name: \"llama3\"
  version: \"3.1\"
  disclosure_flag: true
  disclosure_text_path: \"demo_tech_doc.md\"

risk_management:
  assessment_path: \"demo_risk_assessment.md\"

oversight:
  override_endpoints:
    - \"/api/override\"
    - \"/api/halt\"

logging:
  middleware_active: true

data_governance:
  dataset_path: null
  protected_attributes:
    - \"gender\"
    - \"ethnicity\"

technical_documentation:
  path: \"demo_tech_doc.md\"

banned_models: []

robustness:
  red_team_prompts_path: null
  toxicity_threshold: 0.7
""")

print('demo files written')

## 2. Load the manifest

RaiFlow parses `raiflow.yaml` into a typed `RaiFlowManifest` Pydantic model. Fields not declared in the manifest default gracefully — including the new `technical_documentation` block.

In [ ]:
from raiflow.manifest import load_manifest

manifest = load_manifest('demo_raiflow.yaml')
print(f'System:                {manifest.system_name}')
print(f'Risk level:            {manifest.risk_level}')
print(f'Disclosure flag:       {manifest.model_metadata.disclosure_flag}')
print(f'Logging active:        {manifest.logging.middleware_active}')
print(f'Risk assessment:       {manifest.risk_management.assessment_path}')
print(f'Technical doc path:    {manifest.technical_documentation.path}')
print(f'Override endpoints:    {manifest.oversight.override_endpoints}')

## 3. Run static compliance checks

`CheckRunner` executes all checks for a given pipeline stage and returns `CheckResult` objects.

| Stage | Checks run |
|---|---|
| `pre-commit` | Banned Model Scan |
| `ci` | + Transparency, Risk Mgmt, Human Oversight, Logging, Technical Docs, Bias Detection, Robustness |
| `pre-deploy` | Same as `ci` |
| `post-deploy` | Banned Model Scan, Transparency, Risk Mgmt, Human Oversight, Logging |

Each `CheckResult` has: `article_id`, `rule_id`, `check_name`, `status`, `score`, `threshold`, `remediation_hint`.

In [ ]:
from raiflow.gate import CheckRunner

runner = CheckRunner(manifest)
results = runner.run('ci')

for r in results:
    icon = 'PASS' if r.status == 'pass' else ('SKIP' if r.status == 'skipped' else 'FAIL')
    print(f'[{icon}] [{r.article_id}] {r.check_name} — score={r.score:.2f} threshold={r.threshold:.2f}')
    if r.status == 'fail' and r.remediation_hint:
        print(f'       Fix: {r.remediation_hint}')

## 4. Pipeline stages — pre-commit, ci, pre-deploy, post-deploy

Each stage runs a different subset of checks. `pre-commit` is lightweight (fast, no file I/O). `ci` and `pre-deploy` run the full suite including Article 11.

In [ ]:
for stage in ['pre-commit', 'ci', 'pre-deploy', 'post-deploy']:
    r = CheckRunner(manifest).run(stage)
    rule_ids = [c.rule_id for c in r]
    print(f'{stage:12s} → {len(r)} checks: {rule_ids}')

## 5. `--target` — Article 14 human oversight endpoint scanning

The Article 14 static check scans source files for the endpoint strings declared in `oversight.override_endpoints`. Pass one or more `--target` files via the CLI, or `target_files` in the Python API.

```bash
raiflow check --stage ci --target demo_app.py --no-dashboard
```

In [ ]:
# Without --target: Article 14 fails (missing_evidence)
runner_no_target = CheckRunner(manifest, target_files=[])
art14_no_target = runner_no_target._check_human_oversight()
print(f'No target  → status={art14_no_target.status}, skip_reason={art14_no_target.skip_reason!r}')

# With --target pointing at demo_app.py: endpoints found → pass
runner_with_target = CheckRunner(manifest, target_files=['demo_app.py'])
art14_with_target = runner_with_target._check_human_oversight()
print(f'With target → status={art14_with_target.status}, score={art14_with_target.score}')

## 6. `--threshold` — override per-rule pass thresholds

By default, thresholds come from `eu_ai_act.yaml` per rule (e.g. ART14-5 = 0.90, ART11-1 = 0.90). Pass `--threshold 0.7` to lower the bar for all checks in one run — useful for incremental adoption.

```bash
raiflow check --stage ci --threshold 0.7 --no-dashboard
```

In [ ]:
# Default threshold for ART11-1 from policy
runner_default = CheckRunner(manifest)
print(f'Default ART11-1 threshold: {runner_default._resolve_threshold("ART11-1")}')

# Override to 0.7
runner_override = CheckRunner(manifest, threshold_override=0.7)
print(f'Override ART11-1 threshold: {runner_override._resolve_threshold("ART11-1")}')

# Every CheckResult.threshold reflects the override
results_override = runner_override.run('ci')
thresholds = {r.rule_id: r.threshold for r in results_override}
print(f'All thresholds: {thresholds}')

## 7. `--enable-llm-checks` — semantic evaluation (mocked)

When `--enable-llm-checks` is passed, each upgraded check reads the actual document content and scores it against regulatory criteria using an LLM evaluator. The score (0.0–1.0) is compared against the per-rule threshold to determine pass/fail.

**Upgraded checks:**
- Article 9 — `risk_management_system_check` (threshold 0.85)
- Article 11 — `technical_documentation_check` (threshold 0.90)
- Article 13 — `transparency_by_design_check` (threshold 0.85)
- Article 14 — `intervention_capability_check` (threshold 0.90)

Here we mock the LLM evaluator so this runs without Ollama:

In [ ]:
from unittest.mock import patch, MagicMock
from raiflow.gate import CheckRunner

# Mock the evaluator to return a high score (0.92) for all checks
mock_evaluator = MagicMock()
mock_evaluator.transparency_by_design_check.return_value = 0.92
mock_evaluator.risk_management_system_check.return_value = 0.88
mock_evaluator.technical_documentation_check.return_value = 0.91
mock_evaluator.intervention_capability_check.return_value = 0.93

runner_llm = CheckRunner(
    manifest,
    enable_llm_checks=True,
    target_files=['demo_app.py'],
)

with patch.object(runner_llm, '_load_evaluator', return_value=mock_evaluator):
    results_llm = runner_llm.run('ci')

print('Semantic check results:')
for r in results_llm:
    icon = 'PASS' if r.status == 'pass' else ('SKIP' if r.status == 'skipped' else 'FAIL')
    print(f'[{icon}] [{r.rule_id}] {r.check_name} — score={r.score:.2f} threshold={r.threshold:.2f}')

## 8. LLM unavailable — automatic static fallback

If the LLM call raises any exception (network error, model not loaded, timeout), the check automatically falls back to the static result and appends `'(LLM unavailable — static fallback used)'` to the `remediation_hint`. Your pipeline never breaks due to an LLM outage.

In [ ]:
runner_fallback = CheckRunner(manifest, enable_llm_checks=True)

with patch.object(runner_fallback, '_load_evaluator', side_effect=RuntimeError('Ollama not running')):
    result_fallback = runner_fallback._check_transparency()

print(f'Status:  {result_fallback.status}')
print(f'Score:   {result_fallback.score}')
print(f'Hint:    {result_fallback.remediation_hint}')

## 9. Compliance report — `evaluation_mode` and full schema

`build_report()` produces the JSON artifact written to `raiflow-report.json` by the CLI. It now includes a top-level `evaluation_mode` field: `'semantic'` when `--enable-llm-checks` was used, `'static'` otherwise — useful for audit trails.

In [ ]:
import json
from raiflow.reporter import build_report, write_report

# Static report
report_static = build_report('ci', manifest, results, git_sha='abc123', enable_llm_checks=False)
print(f"evaluation_mode (static): {report_static['evaluation_mode']}")

# Semantic report
report_semantic = build_report('ci', manifest, results_llm, git_sha='abc123', enable_llm_checks=True)
print(f"evaluation_mode (semantic): {report_semantic['evaluation_mode']}")

print(f"\nTop-level fields: {list(report_semantic.keys())}")
print(f"Overall status:   {report_semantic['overall_status']}")
print(f"\nChecks:")
for c in report_semantic['checks']:
    print(f"  {c['status'].upper():8s} {c['rule_id']:12s} score={c['score']:.2f}  {c['check_name']}")

## 10. Score and risk level

RaiFlow computes an overall compliance score and maps it to a risk level — the same values shown in the dashboard ScoreBanner.

In [ ]:
def compliance_summary(results):
    passing = sum(1 for r in results if r.status == 'pass')
    failing = sum(1 for r in results if r.status == 'fail')
    skipped = sum(1 for r in results if r.status == 'skipped')
    non_skipped = passing + failing
    score = round((passing / non_skipped) * 100) if non_skipped else None
    def risk(s):
        if s is None: return 'Unknown'
        if s == 100:  return 'Low'
        if s >= 75:   return 'Medium'
        if s >= 50:   return 'High'
        return 'Critical'
    print(f'Passing: {passing}  Failing: {failing}  Skipped: {skipped}')
    print(f'Score:   {score}%  →  Risk: {risk(score)}')

print('Static run:')
compliance_summary(results)
print('\nSemantic run (mocked LLM):')
compliance_summary(results_llm)

## 11. `raiflow generate-tests` — auto-generate pytest compliance tests

`raiflow generate-tests` reads `eu_ai_act.yaml` and generates a pytest file for every rule, so you can run compliance checks as part of your normal test suite.

```bash
raiflow generate-tests
# custom policy and output directory:
raiflow generate-tests --policy policies/eu_ai_act.yaml --output-dir tests/generated
```

Let's run it programmatically:

In [ ]:
import importlib.resources
from raiflow.generator import TestGenerator

policy_ref = importlib.resources.files('raiflow.data').joinpath('eu_ai_act.yaml')
gen = TestGenerator(str(policy_ref))
gen.generate('demo_generated_tests')

generated = list(Path('demo_generated_tests').glob('*.py'))
print(f'Generated {len(generated)} test file(s):')
for f in generated:
    lines = f.read_text().splitlines()
    test_fns = [l.strip() for l in lines if l.strip().startswith('def test_')]
    print(f'  {f.name} — {len(test_fns)} tests')

## 12. CLI reference

All commands and flags at a glance:

```bash
# --- raiflow init ---
raiflow init                          # scaffold manifest + workflow
raiflow init --directory ./my-project # target a specific directory
raiflow init --force                  # overwrite existing files

# --- raiflow check ---
raiflow check --stage ci --no-dashboard                    # headless static run
raiflow check --stage pre-commit --no-dashboard            # lightweight pre-commit
raiflow check --stage pre-deploy --no-dashboard            # full pre-deploy gate
raiflow check --stage ci --target app.py --no-dashboard    # Article 14 endpoint scan
raiflow check --stage ci --threshold 0.7 --no-dashboard    # lower threshold for all rules
raiflow check --stage ci --enable-llm-checks --no-dashboard  # semantic LLM evaluation
raiflow check --stage ci --dry-run --no-dashboard          # always exit 0 (reporting only)
raiflow check --stage ci --output my-report.json           # custom report path
raiflow check --stage ci                                   # opens browser dashboard
raiflow check --stage ci --dashboard --dashboard-port 9000 # custom port

# --- raiflow generate-tests ---
raiflow generate-tests                                     # write to tests/generated/
raiflow generate-tests --output-dir tests/compliance       # custom output dir
```

## 13. GitHub Actions integration

`raiflow init` drops `.github/workflows/rai-compliance.yml` into your repo. It defines four jobs:

| Job | Stage | Blocks on failure |
|---|---|---|
| `pre-commit-checks` | `pre-commit` | PR merge |
| `compliance-gate` | `ci` | Build |
| `build-and-sign` | — | Produces signed artifact manifest |
| `deploy-gate` | `pre-deploy` | Deployment |

```yaml
# .github/workflows/rai-compliance.yml (excerpt)
- name: Run compliance gate
  run: raiflow check --stage ci --no-dashboard

# With LLM checks (requires OLLAMA_HOST or API key secret):
- name: Run semantic compliance gate
  run: raiflow check --stage ci --enable-llm-checks --no-dashboard
  env:
    GEMMA_API_KEY: ${{ secrets.GEMMA_API_KEY }}
```

In CI environments (`CI=true`), the browser dashboard is automatically suppressed — no flags needed.

## 14. Cleanup

In [ ]:
import shutil

for f in ['demo_raiflow.yaml', 'demo_risk_assessment.md', 'demo_tech_doc.md', 'demo_app.py']:
    Path(f).unlink(missing_ok=True)
shutil.rmtree('demo_generated_tests', ignore_errors=True)
print('Done.')